---
layout: csa-pastel
toc: true
title: "4.17 Recursive Searching and Sorting"
description: "Trace recursive collections, binary search, and merge sort with visual traces and partner activities."
author: Meryl
categories: [CSA]
permalink: /csa/unit4/4.17-recursive-searching-and-sorting/
---


# 4.17 — Recursive Searching and Sorting

**AP CSA · Unit 4: Data Collections**

How can repeatedly solving a smaller problem help us search a sorted collection or put an unsorted collection in order?

## Learning targets

Trace recursion over Strings, arrays, and ArrayLists; track the bounds of recursive binary search; and show both the splitting and merging stages of merge sort.

**Before you start:** Topic 4.16 recursion (base cases, recursive calls, returning to the caller), array indexes, ArrayList `size()`/`get()`, and comparison-based searching/sorting.

**Try it:** Predict the answer, record your work in your notes, and compare with a partner. Open each answer reveal after attempting the activity.


## 1. Warm-up: calls go down, returns come back

```java
public static String reverse(String word) {
    if (word.length() <= 1) {
        return word;
    }
    return reverse(word.substring(1)) + word.substring(0, 1);
}
```

**Predict:** What does `reverse("JAVA")` return? What is waiting to happen while each recursive call runs?

Fill in the missing returns:

```text
reverse("JAVA") = reverse("AVA") + "J"
reverse("AVA")  = reverse("VA")  + "A"
reverse("VA")   = reverse("A")   + "V"
reverse("A")    = ?

Returning upward: "A" -> ? -> ? -> ?
```

<details markdown="1">
<summary>Check your reasoning</summary>

`reverse("A")` returns `"A"`. Returning upward produces `"AV"`, then `"AVA"`, then `"AVAJ"`.

Each caller waits to append its first character until the smaller call has returned. The problem shrinks because `substring(1)` removes the first character. Empty and one-character Strings reach the base case immediately.

</details>

**Partner variation:** Move the first character before the recursive call in the returned expression. Predict the result before tracing: `word.substring(0, 1) + reverse(word.substring(1))` would preserve the original order.


## 2. Recursion over arrays and ArrayLists

An index can describe the part of a collection still to process.

```java
// Precondition: values is nonnull and 0 <= index <= values.length.
public static int sumFrom(int[] values, int index) {
    if (index == values.length) {
        return 0;
    }
    return values[index] + sumFrom(values, index + 1);
}
```

For `int[] values = {4, 1, 3};`, complete this trace of `sumFrom(values, 0)`:

| index | Pending calculation | Returned value |
|---|---|---|
| 0 | `4 + sumFrom(values, 1)` | … |
| 1 | `1 + sumFrom(values, 2)` | … |
| 2 | `3 + sumFrom(values, 3)` | … |
| 3 | Base case | … |

Now consider an ArrayList method. Add `import java.util.ArrayList;` above the surrounding class if running it.

```java
// Precondition: words is nonnull, contains no null elements,
// and 0 <= index <= words.size().
public static int countLong(ArrayList<String> words, int index) {
    if (index == words.size()) {
        return 0;
    }
    int current = 0;
    if (words.get(index).length() >= 4) {
        current = 1;
    }
    return current + countLong(words, index + 1);
}
```

**Think-pair-share:** For a list containing `"cat", "java", "loops"`, what does `countLong(words, 0)` return? What replaces array `.length` and `[index]` in the list version?

<details markdown="1">
<summary>Check your reasoning</summary>

The array returns are `0` at index 3, then `3`, `4`, and finally `8` at index 0. The ArrayList result is `2` because `"java"` and `"loops"` qualify.

ArrayList uses `.size()` for the element count and `.get(index)` for indexed access. Both methods must check the base case before accessing an element. An empty collection called with index `0` returns `0`.

</details>


## 3. Binary search: keep only the possible region

Start with this **ascending sorted** array:

```text
index:  0   1   2   3   4   5   6
value:  3   7  12  18  24  31  42
```

`low` and `high` are inclusive endpoints. Compare the target with the middle value:

- Equal: return the middle index.
- Smaller: continue strictly to the left of the middle.
- Larger: continue strictly to the right of the middle.
- No indexes remain (`low > high`): return `-1`.

**Why sorted matters:** Once we see `18`, we can rule out the left side for target `31` because every value on that side is at most `18`. That guarantee disappears in an unsorted array.

```java
// Precondition: values is nonnull and sorted in ascending order.
// Initial call: binarySearch(values, target, 0, values.length - 1).
public static int binarySearch(int[] values, int target, int low, int high) {
    if (low > high) {
        return -1;
    }
    int mid = low + (high - low) / 2;
    if (values[mid] == target) {
        return mid;
    }
    if (target < values[mid]) {
        return binarySearch(values, target, low, mid - 1);
    }
    return binarySearch(values, target, mid + 1, high);
}
```

Integer division rounds down here because the difference is nonnegative. Each unsuccessful comparison removes the already-checked middle index. The base-case check occurs before indexing the array.

**Think about it:** Why do recursive calls need `return`? Each call must pass the found index or `-1` back to its caller.


## 4. Human binary search

Put the seven values on cards. With your group, assign one person to track `low`, another to track `high`, and a third to compute `mid`. Predict which cards you can discard after each comparison.

### Round A: find 31

| Call | low | high | mid | values[mid] | Next action |
|---|---|---|---|---|---|
| 1 | 0 | 6 | … | … | … |
| 2 | … | … | … | … | … |

### Round B: find 20

Start again from the full array. Continue until the method returns, including the final empty-range call.

<details markdown="1">
<summary>Check both traces</summary>

**Target 31:**

| Call | low | high | mid | values[mid] | Next action |
|---|---|---|---|---|---|
| 1 | 0 | 6 | 3 | 18 | Search indexes 4–6 |
| 2 | 4 | 6 | 5 | 31 | Return index 5 |

**Target 20:**

| Call | low | high | mid | values[mid] | Next action |
|---|---|---|---|---|---|
| 1 | 0 | 6 | 3 | 18 | Search indexes 4–6 |
| 2 | 4 | 6 | 5 | 31 | Search index 4 |
| 3 | 4 | 4 | 4 | 24 | Search indexes 4–3 |
| 4 | 4 | 3 | Not computed | Not accessed | Return -1 |

Round B makes **three middle-value inspections** and **four method calls**, including the empty-range base case. Always state what you are counting.

</details>

### Change one condition

1. If the array has duplicate targets, must this method return the first matching index?
2. If the array is empty, what happens on the initial call?
3. Is the recursive implementation the only way to perform binary search?

<details markdown="1">
<summary>Check your reasoning</summary>

1. No. It returns a matching middle index, which need not be the first occurrence.
2. `low = 0` and `high = -1`; the base case returns `-1` without accessing an element.
3. No. A loop can update the same bounds iteratively.

</details>


### Your response

- My Round B trace:
- Why the array must be sorted:
- Why `mid + 1` or `mid - 1` makes progress:
- Why three inspections can involve four calls:


## 5. Merge sort: splitting is only half the job

Binary search eliminates a region that cannot contain the target. Merge sort processes **both halves** and then combines their sorted results.

Use these cards: **8, 3, 6, 2, 7, 1, 5, 4**.

### Split phase

```text
                 [8, 3, 6, 2, 7, 1, 5, 4]
                 /                       \
          [8, 3, 6, 2]                [7, 1, 5, 4]
          /          \                /          \
       [8, 3]      [6, 2]          [7, 1]      [5, 4]
       /   \       /   \           /   \       /   \
      [8]  [3]    [6]  [2]         [7]  [1]    [5]  [4]
```

A zero- or one-element collection is already sorted. Splitting alone does not rearrange the values into order.

### Merge phase — your turn

Fill each blank before checking:

```text
[8] + [3] -> [____]        [6] + [2] -> [____]
[7] + [1] -> [____]        [5] + [4] -> [____]

[3, 8] + [2, 6] -> [____________]
[1, 7] + [4, 5] -> [____________]

[2, 3, 6, 8] + [1, 4, 5, 7] -> [________________________]
```

At each merge, compare the first unused element of each sorted half. Take the smaller one and advance only that half's pointer. When a half runs out, append the other half's remaining elements.

<details markdown="1">
<summary>Check the merge stages</summary>

Pairs: `[3, 8]`, `[2, 6]`, `[1, 7]`, `[4, 5]`.

Groups of four: `[2, 3, 6, 8]` and `[1, 4, 5, 7]`.

Final result: `[1, 2, 3, 4, 5, 6, 7, 8]`.

The diagram groups results of the same size. A typical recursive implementation finishes the left half before recursively sorting the right half; it does not necessarily execute all same-sized merges together.

</details>


## 6. Partner activity: merge with two pointers

Partner A manages `[2, 3, 6, 8]`; partner B manages `[1, 4, 5, 7]`. Take turns announcing the comparison and moving one pointer.

| Step | Next left value | Next right value | Value appended |
|---|---|---|---|
| 1 | 2 | 1 | … |
| 2 | 2 | 4 | … |
| 3 | … | … | … |

**Challenge:** Count only comparisons between two available values, not loop-condition checks. How many such comparisons occur in this final merge?

<details markdown="1">
<summary>Check the complete merge trace</summary>

| Step | Left | Right | Appended |
|---|---|---|---|
| 1 | 2 | 1 | 1 |
| 2 | 2 | 4 | 2 |
| 3 | 3 | 4 | 3 |
| 4 | 6 | 4 | 4 |
| 5 | 6 | 5 | 5 |
| 6 | 6 | 7 | 6 |
| 7 | 8 | 7 | 7 |
| 8 | 8 | Exhausted | 8 |

There are **7 comparisons between available values**. Appending the leftover `8` needs no comparison against a right-side value.

</details>

### Read the merge helper

This method combines two already-sorted arrays into a new sorted array. It is the combine step, not a complete merge sort.

```java
public static int[] merge(int[] left, int[] right) {
    int[] result = new int[left.length + right.length];
    int i = 0;
    int j = 0;
    int k = 0;
    while (i < left.length && j < right.length) {
        if (left[i] <= right[j]) {
            result[k] = left[i];
            i++;
        } else {
            result[k] = right[j];
            j++;
        }
        k++;
    }
    while (i < left.length) {
        result[k] = left[i];
        i++;
        k++;
    }
    while (j < right.length) {
        result[k] = right[j];
        j++;
        k++;
    }
    return result;
}
```

**Discuss:** Why do we need both leftover loops? What would go wrong if the first loop used `||` instead of `&&`?

<details markdown="1">
<summary>Check your reasoning</summary>

Either half can run out first. The leftover loops copy everything still unconsumed in the other half. With `||`, the first loop could continue after one side runs out and attempt an invalid array access. The `<=` tie rule chooses the left element first when values are equal.

</details>


## 7. Compare the algorithms

| Algorithm | Starting condition | What happens to the data? | How the work shrinks |
|---|---|---|---|
| Linear search | May be unsorted | Finds a match by inspection | Checks one more element at a time |
| Binary search | Sorted using the comparison order | Finds a matching index; does not sort | Continues in one half |
| Merge sort | May be unsorted | Produces sorted order | Sorts both halves, then merges |

With 15 sorted elements, this binary-search implementation needs at most **4 middle-value inspections**; a linear search may inspect all 15. Binary search's benefit assumes the data is already sorted. Sorting an unsorted collection solely for one search adds work.


## 8. Popcorn hack: diagnose the recursion

A classmate changes the right-side binary-search call to:

```java
return binarySearch(values, target, mid, high);
```

Use `values = {3, 7}` and `target = 7`.

1. Write `low`, `high`, and `mid` for the first two calls.
2. Explain why the range can fail to shrink.
3. Repair the call.
4. Design a missing-target test and show the expected result.

<details markdown="1">
<summary>Reveal a solution after your trace</summary>

The first call has `low = 0`, `high = 1`, `mid = 0`. The faulty call repeats the same bounds: `0, 1`, so `mid` remains `0`. It keeps inspecting `3` and never reaches the `7`; eventually Java exhausts its call stack.

Repair:

```java
return binarySearch(values, target, mid + 1, high);
```

For a missing target such as `5`, the repaired algorithm inspects index `0`, then index `1`, then reaches bounds `1, 0` and returns `-1`.

</details>

### Merge-sort transfer challenge

Trace merge sort on `[9, 2, 6, 1]`. Show the singleton groups, sorted pairs, and final merge. Then repeat the merge step for sorted halves `[2, 4]` and `[2, 3]`, preserving both copies of `2`.

<details markdown="1">
<summary>Check the transfer challenge</summary>

Singletons: `[9]`, `[2]`, `[6]`, `[1]`. Sorted pairs: `[2, 9]`, `[1, 6]`. Final merge: `[1, 2, 6, 9]`.

Duplicate example: `[2, 2, 3, 4]`. Sorting retains duplicates; it does not remove them. The helper's tie rule takes the left `2` first.

</details>


### Your response

- Faulty-call trace:
- My repair and explanation:
- Missing-target test and expected result:
- Merge-sort split and merge stages:
- My rule for recognizing a valid base case:


## 9. Exit ticket

1. Trace binary search for `12` in `[3, 7, 12, 18, 24, 31, 42]`. List middle indexes in order, then the returned index.
2. What happens when `low > high`? Is another middle element inspected?
3. Merge sorted halves `[1, 5, 9]` and `[2, 4, 8]`.
4. Why does merge sort need a merge phase after splitting?
5. For the warm-up method, what does `reverse("")` return?

<details markdown="1">
<summary>Exit-ticket key</summary>

1. Middle indexes `3`, `1`, `2`; return `2`.
2. The method returns `-1` before computing or accessing another middle element.
3. `[1, 2, 4, 5, 8, 9]`.
4. The split creates small subproblems but does not put values from different halves in order. Merging combines sorted halves into a larger sorted result.
5. The empty String; it already satisfies the base case.

</details>
